# Preprocessing — Member 4: Data Validation, Duplicates & Train/Test Split
## Online Shopper Purchase Likelihood Prediction System
**IT3051 – Fundamentals of Data Mining | Mini Project 2026 | Group DS_WE_01.02**

This is the first step in the preprocessing relay. Nothing else in the pipeline happens before this notebook's
steps are complete: data validation, removing confirmed duplicates (`df_clean`), and the train/test split.

**Prediction point definition:** the prediction is made at any point during an active session, before checkout is
completed — using only information generated by the visitor's browsing behaviour up to that moment.


## Setup

In [1]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42

df = pd.read_csv("../data/online_shoppers_intention.csv")
df.shape


(12330, 18)

## Invalid-value validation
Separate from duplicates: check for values that would be impossible given each column's meaning, rather than
assuming the dataset is clean just because EDA found no missing values.

In [2]:
count_duration_cols = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
                       "ProductRelated", "ProductRelated_Duration"]
rate_cols = ["BounceRates", "ExitRates"]

issues = {}
issues["negative counts/durations"] = (df[count_duration_cols] < 0).sum().sum()
issues["BounceRates/ExitRates outside [0,1]"] = (~df[rate_cols].apply(lambda s: s.between(0, 1))).sum().sum()
issues["SpecialDay outside [0,1]"] = (~df["SpecialDay"].between(0, 1)).sum()
issues["PageValues negative"] = (df["PageValues"] < 0).sum()

expected_months = {"Feb","Mar","May","June","Jul","Aug","Sep","Oct","Nov","Dec"}
expected_visitor = {"Returning_Visitor", "New_Visitor", "Other"}
issues["unexpected Month values"] = (~df["Month"].isin(expected_months)).sum()
issues["unexpected VisitorType values"] = (~df["VisitorType"].isin(expected_visitor)).sum()
issues["unexpected Weekend values"] = (~df["Weekend"].isin([True, False])).sum()

for k, v in issues.items():
    print(f"{k}: {v}")


negative counts/durations: 0
BounceRates/ExitRates outside [0,1]: 0
SpecialDay outside [0,1]: 0
PageValues negative: 0
unexpected Month values: 0
unexpected VisitorType values: 0
unexpected Weekend values: 0


No invalid values were found in any of the checks above — every count, duration, rate, and category falls
within its expected range. This confirms the dataset is valid, not just free of missing values, and this check
is documented explicitly here to satisfy the "invalid records" part of the rubric, separate from the
duplicate-record handling below.

## Removing confirmed duplicates — `df_clean`
The EDA notebook's duplicate bias check already confirmed the 125 duplicate rows have a purchase rate very close
to the full dataset's rate, so removing them will not meaningfully shift the class balance. Duplicates are
removed **before** the train/test split, so identical rows cannot end up split across both training and test
sets, which would otherwise produce overly optimistic, misleading evaluation results.

In [3]:
df_clean = df.drop_duplicates().reset_index(drop=True)
print("Original shape:", df.shape)
print("df_clean shape:", df_clean.shape)
print("Rows removed:", df.shape[0] - df_clean.shape[0])


Original shape: (12330, 18)
df_clean shape: (12205, 18)
Rows removed: 125


**Justification (Member 4):** validating for invalid values before touching duplicates keeps the two checks
separate and explicit, as the rubric asks for both. `df_clean` is used for every step from here onward — `df`
itself is left untouched.

## Train/test split

In [4]:
from sklearn.model_selection import train_test_split

TARGET = "Revenue"
FEATURE_COLS = [c for c in df_clean.columns if c != TARGET]

X = df_clean[FEATURE_COLS].copy()
y = df_clean[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)
print("Train purchase rate: {:.3f}   Test purchase rate: {:.3f}".format(y_train.mean(), y_test.mean()))


Train shape: (9764, 17)  Test shape: (2441, 17)
Train purchase rate: 0.156   Test purchase rate: 0.156


**Justification (Member 4):** an 80/20 split, stratified on `Revenue`, keeps the same ~15.5% purchase rate
in both sets — important given the class imbalance, since a non-stratified split could easily leave the test set
with a meaningfully different (and less reliable) class balance.

In [5]:
X_train.to_csv("../data/X_train.csv", index=False)
X_test.to_csv("../data/X_test.csv", index=False)
y_train.to_csv("../data/y_train.csv", index=False)
y_test.to_csv("../data/y_test.csv", index=False)

print("Saved X_train, X_test, y_train, y_test to the data/ folder for the next member to load.")


Saved X_train, X_test, y_train, y_test to the data/ folder for the next member to load.


---
## Handover to Member 1

The next person in the relay (Member 1 — feature engineering and outlier/skew treatment) should:
1. Pull this branch's changes after it's merged into `develop`.
2. Load `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv` from the `data/` folder — **not** the original
   `online_shoppers_intention.csv` — since these are already validated, deduplicated, and split.
3. Continue building on a new notebook (or a new section of the shared one), fitting everything on the training
   set only, exactly as planned.
